# 📚 Book Recommendation System — ML Notebook
---
**Techniques:** Content-Based Filtering · Collaborative Filtering  
**Algorithm:** Cosine Similarity (TF-IDF vectors + user-item matrix)  
**Output:** `similarity_matrix.pkl`, `books_data.pkl`, `pivot_table.pkl`

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('muted')
print("✅ Libraries imported.")

## 2. Create Sample Dataset

We build a realistic sample dataset with columns:
`user_id, book_title, rating, author, genre, description`

In [ ]:
# ── Sample dataset: 30 books, 10 users, explicit ratings (1–5) ─────────────
data = {
    'user_id': [
        1,1,1,1,1,
        2,2,2,2,2,
        3,3,3,3,3,
        4,4,4,4,4,
        5,5,5,5,5,
        6,6,6,7,7,
        7,8,8,8,9,
        9,9,10,10,10,
    ],
    'book_title': [
        '1984','Brave New World','Fahrenheit 451','The Road','Animal Farm',
        'Harry Potter and the Sorcerer\'s Stone','The Hobbit','The Lord of the Rings','Eragon','Percy Jackson',
        'The Da Vinci Code','Angels and Demons','Inferno','Digital Fortress','The Girl with the Dragon Tattoo',
        'To Kill a Mockingbird','Of Mice and Men','The Catcher in the Rye','Lord of the Flies','The Outsiders',
        'A Brief History of Time','The Selfish Gene','Sapiens','Cosmos','The Grand Design',
        'The Alchemist','Siddhartha','The Prophet','Dune','Foundation',
        '1984','Sapiens','The Alchemist','Dune','Harry Potter and the Sorcerer\'s Stone',
        'Fahrenheit 451','The Hobbit','Foundation','The Da Vinci Code','To Kill a Mockingbird',
    ],
    'rating': [
        5,4,5,3,4,
        5,5,4,4,3,
        5,4,4,3,4,
        5,4,4,3,3,
        5,5,5,4,4,
        5,4,5,5,4,
        4,5,5,4,4,
        5,5,4,5,5,
    ],
    'author': [
        'George Orwell','Aldous Huxley','Ray Bradbury','Cormac McCarthy','George Orwell',
        'J.K. Rowling','J.R.R. Tolkien','J.R.R. Tolkien','Christopher Paolini','Rick Riordan',
        'Dan Brown','Dan Brown','Dan Brown','Dan Brown','Stieg Larsson',
        'Harper Lee','John Steinbeck','J.D. Salinger','William Golding','S.E. Hinton',
        'Stephen Hawking','Richard Dawkins','Yuval Noah Harari','Carl Sagan','Stephen Hawking',
        'Paulo Coelho','Hermann Hesse','Kahlil Gibran','Frank Herbert','Isaac Asimov',
        'George Orwell','Yuval Noah Harari','Paulo Coelho','Frank Herbert','J.K. Rowling',
        'Ray Bradbury','J.R.R. Tolkien','Isaac Asimov','Dan Brown','Harper Lee',
    ],
    'genre': [
        'Dystopia','Dystopia','Dystopia','Post-Apocalyptic','Political Satire',
        'Fantasy','Fantasy','Fantasy','Fantasy','Fantasy',
        'Thriller','Thriller','Thriller','Thriller','Mystery',
        'Classic','Classic','Classic','Classic','Classic',
        'Science','Science','Non-Fiction','Science','Science',
        'Philosophy','Philosophy','Philosophy','Sci-Fi','Sci-Fi',
        'Dystopia','Non-Fiction','Philosophy','Sci-Fi','Fantasy',
        'Dystopia','Fantasy','Sci-Fi','Thriller','Classic',
    ],
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Unique books : {df['book_title'].nunique()}")
print(f"Unique users : {df['user_id'].nunique()}")
df.head(8)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating distribution
df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white', rwidth=0.8)
axes[0].set_title('Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Genre distribution
df.drop_duplicates('book_title')['genre'].value_counts().plot(
    kind='barh', ax=axes[1], color='#e07b5c', edgecolor='white')
axes[1].set_title('Books per Genre', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4. Content-Based Filtering

**Idea:** Represent each book as a text vector (author + genre) and compute  
cosine similarity between all pairs. Books close in vector space share similar characteristics.

In [ ]:
# ── Build a unique book profile ─────────────────────────────────────────────
books_df = df.drop_duplicates('book_title')[['book_title', 'author', 'genre']].reset_index(drop=True)

# Combine author and genre into a single 'tags' string for TF-IDF
books_df['tags'] = books_df['author'].str.lower() + ' ' + books_df['genre'].str.lower()

print(f"Unique books: {len(books_df)}")
books_df.head()

In [ ]:
# ── TF-IDF Vectorization ────────────────────────────────────────────────────
# TF-IDF converts text to numerical vectors, weighing important/unique terms higher.
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books_df['tags'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print("  rows = books, cols = unique terms in vocabulary")

In [ ]:
# ── Compute Cosine Similarity between all book pairs ────────────────────────
content_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Wrap in a DataFrame for readability
content_sim_df = pd.DataFrame(
    content_similarity,
    index=books_df['book_title'],
    columns=books_df['book_title']
)
print(f"Similarity matrix shape: {content_sim_df.shape}")
content_sim_df.iloc[:5, :5]

## 5. Collaborative Filtering (Item-Based)

**Idea:** Build a user-book rating matrix and find books that are rated  
similarly by the same users using cosine similarity on rating vectors.

In [ ]:
# ── Build user-book pivot table ──────────────────────────────────────────────
pivot_table = df.pivot_table(index='book_title', columns='user_id', values='rating')
pivot_table.fillna(0, inplace=True)

print(f"Pivot table shape: {pivot_table.shape}")
print("  rows = books, cols = users, values = ratings (0 = not rated)")
pivot_table.head()

In [ ]:
# ── Compute Collaborative Similarity ────────────────────────────────────────
collab_similarity = cosine_similarity(pivot_table)

collab_sim_df = pd.DataFrame(
    collab_similarity,
    index=pivot_table.index,
    columns=pivot_table.index
)
print(f"Collaborative similarity matrix: {collab_sim_df.shape}")

## 6. Hybrid Similarity

We combine content-based and collaborative scores for better recommendations:  
`hybrid = 0.5 × content + 0.5 × collaborative`

In [ ]:
# ── Align both matrices to the same book index ──────────────────────────────
common_books = books_df['book_title'].tolist()

# Reindex collaborative matrix to match content matrix index
collab_aligned = collab_sim_df.reindex(index=common_books, columns=common_books).fillna(0)
content_aligned = content_sim_df.reindex(index=common_books, columns=common_books).fillna(0)

# Weighted hybrid
CONTENT_WEIGHT = 0.5
COLLAB_WEIGHT  = 0.5
hybrid_sim = (CONTENT_WEIGHT * content_aligned.values) + (COLLAB_WEIGHT * collab_aligned.values)

hybrid_sim_df = pd.DataFrame(hybrid_sim, index=common_books, columns=common_books)
print(f"Hybrid similarity matrix shape: {hybrid_sim_df.shape}")

## 7. Recommendation Function

In [ ]:
def recommend_books(book_name, n=5, method='hybrid'):
    """
    Recommend books similar to the given book title.

    Parameters
    ----------
    book_name : str   — Title of the input book (case-insensitive partial match supported)
    n         : int   — Number of recommendations to return
    method    : str   — 'content', 'collaborative', or 'hybrid'

    Returns
    -------
    list of dicts with title, author, genre, similarity_score
    """
    # ── Select similarity matrix based on method ─────────────────────────────
    sim_map = {
        'content'       : content_aligned,
        'collaborative' : collab_aligned,
        'hybrid'        : hybrid_sim_df,
    }
    sim_matrix = sim_map.get(method, hybrid_sim_df)

    # ── Case-insensitive lookup with partial match ────────────────────────────
    all_titles = list(sim_matrix.index)
    matched = [t for t in all_titles if book_name.lower() in t.lower()]

    if not matched:
        print(f"⚠️  '{book_name}' not found. Available titles:\n")
        for t in sorted(all_titles): print(f"   • {t}")
        return []

    exact_title = matched[0]  # use first match

    # ── Get top-N similar books (exclude the book itself) ────────────────────
    scores = sim_matrix[exact_title].sort_values(ascending=False)
    top_n  = scores.iloc[1:n+1]  # skip index 0 (itself)

    # ── Enrich with metadata ─────────────────────────────────────────────────
    results = []
    for title, score in top_n.items():
        info = books_df[books_df['book_title'] == title]
        if not info.empty:
            results.append({
                'title'           : title,
                'author'          : info['author'].values[0],
                'genre'           : info['genre'].values[0],
                'similarity_score': round(float(score), 4),
            })

    return results


def display_recommendations(book_name, n=5, method='hybrid'):
    """Pretty-print recommendations."""
    print(f"\n{'═'*60}")
    print(f"  📚  Books similar to: '{book_name}'  [{method}]")
    print(f"{'═'*60}")
    recs = recommend_books(book_name, n, method)
    for i, r in enumerate(recs, 1):
        print(f"  {i}. {r['title']}")
        print(f"     ✍  {r['author']}  |  🏷  {r['genre']}  |  Score: {r['similarity_score']}")
    print(f"{'─'*60}")
    return recs

In [ ]:
# ── Test the recommender ────────────────────────────────────────────────────
display_recommendations('1984')
display_recommendations('Harry Potter', method='content')
display_recommendations('Sapiens', method='collaborative')

## 8. Visualise Recommendations

In [ ]:
def plot_recommendations(book_name, n=5):
    recs = recommend_books(book_name, n)
    if not recs: return
    titles = [r['title'][:35] + '…' if len(r['title']) > 35 else r['title'] for r in recs]
    scores = [r['similarity_score'] for r in recs]
    colors = plt.cm.Greens(np.linspace(0.4, 0.85, len(titles)))
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(titles[::-1], scores[::-1], color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Similarity Score')
    ax.set_title(f"Recommendations for '{book_name}'", fontweight='bold')
    ax.set_xlim(0, max(scores) * 1.35)
    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{score:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

plot_recommendations('1984')

## 9. Save Model Artifacts

In [ ]:
# Save artifacts for the FastAPI backend
import os
os.makedirs('../backend', exist_ok=True)

artifacts = {
    '../backend/hybrid_similarity.pkl' : hybrid_sim_df,
    '../backend/books_data.pkl'        : books_df,
    '../backend/pivot_table.pkl'       : pivot_table,
}

for path, obj in artifacts.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"  ✅ Saved: {path}")

print("\n🎉 All model artifacts saved to backend/")

## 10. Conclusion

| Method | Strength | Weakness |
|---|---|---|
| Content-Based | Works for new books | Ignores community preferences |
| Collaborative | Captures user taste | Cold-start for new users |
| **Hybrid** | **Best of both worlds** | Requires both rating + metadata |

**Next Steps:** Deploy via FastAPI → React frontend for a full web application.